# AgentWeather: Exploratory Data Analysis (EDA)
**City Target:** Islamabad, Pakistan  
**Coverage:** 8 Months of Hourly Observations (~5,800 records)  
**Primary Metrics Analyzed:**
1. Air Quality Index (AQI) & Pollutants (`pm2_5`, `pm10`, `no2`, `so2`, `co`, `o3`)
2. Temperature (`temp_c`)
3. Precipitation (`precip_mm`) & Rain Probability (`rain_prob`)
4. Relative Humidity (`humidity_pct`)
5. UV Radiation (`uv_index`)
6. Wind Speed (`wind_speed`) & Circular Direction (`wind_sin`, `wind_cos`)


In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set visual styles
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (12, 6)

# Load historical features
data_path = Path("../data/historical_features.parquet")
df = pd.read_parquet(data_path)
df["timestamp"] = pd.to_datetime(df["timestamp"])
print("Dataset Shape:", df.shape)
print("Date Range:", df["timestamp"].min(), "to", df["timestamp"].max())
df.head()


## 1. Missing Data & Imputation Audit
Verification that all 1-3 hour gaps were resolved via forward fill + linear interpolation,
and longer gaps via 7-day diurnal hour-of-day averaging.


In [ ]:
feature_cols = [c for c in df.columns if not c.startswith("target_")]
null_counts = df[feature_cols].isnull().sum()
print("Total nulls in feature columns:", null_counts.sum())
assert null_counts.sum() == 0, "Error: Unexpected nulls found!"
print("Imputation check PASSED: Exactly 0 missing values.")


## 2. Statistical Distributions Across 6 Target Categories
Visualizing probability densities, hazard thresholds, and skewness across all core weather and air quality variables.


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
sns.histplot(df["aqi"], kde=True, ax=axes[0, 0], color="#e74c3c", bins=30)
axes[0, 0].axvline(150, color="darkred", linestyle="--", label="Hazard (>150)")
axes[0, 0].set_title("AQI Distribution")
axes[0, 0].legend()

sns.histplot(df["temp_c"], kde=True, ax=axes[0, 1], color="#e67e22", bins=30)
axes[0, 1].axvline(40, color="darkred", linestyle="--", label="Heat Hazard (>40°C)")
axes[0, 1].set_title("Temperature Distribution (°C)")
axes[0, 1].legend()

sns.histplot(df[df["precip_mm"] > 0]["precip_mm"], kde=True, ax=axes[0, 2], color="#3498db", bins=20)
axes[0, 2].set_title("Precipitation (mm, wet hours)")

sns.histplot(df["humidity_pct"], kde=True, ax=axes[1, 0], color="#2ecc71", bins=30)
axes[1, 0].axvline(85, color="darkred", linestyle="--", label="Hazard (>85%)")
axes[1, 0].set_title("Humidity Distribution (%)")
axes[1, 0].legend()

sns.histplot(df[df["uv_index"] > 0]["uv_index"], kde=True, ax=axes[1, 1], color="#9b59b6", bins=15)
axes[1, 1].axvline(8, color="darkred", linestyle="--", label="Hazard (>=8)")
axes[1, 1].set_title("UV Index Distribution")
axes[1, 1].legend()

sns.histplot(df["wind_speed"], kde=True, ax=axes[1, 2], color="#34495e", bins=30)
axes[1, 2].axvline(50, color="darkred", linestyle="--", label="Hazard (>50 km/h)")
axes[1, 2].set_title("Wind Speed Distribution (km/h)")
axes[1, 2].legend()

plt.tight_layout()
plt.show()


## 3. Diurnal Seasonality Patterns (24-Hour Day/Night Cycles)
Evaluating diurnal periodicity across temperature, air quality, humidity, UV radiation, and wind velocity.


In [ ]:
hour_grp = df.groupby("hour")
plt.figure(figsize=(12, 5))
plt.plot(hour_grp["temp_c"].mean().index, hour_grp["temp_c"].mean().values, marker="o", color="#e67e22", label="Mean Temp (°C)")
plt.plot(hour_grp["aqi"].mean().index, hour_grp["aqi"].mean().values, marker="s", color="#e74c3c", label="Mean AQI")
plt.title("Diurnal Cycle: Temperature & AQI vs Hour of Day (UTC)")
plt.xlabel("Hour of Day (UTC)")
plt.legend()
plt.grid(True)
plt.show()


## 4. Correlation with 24-Hour Future Targets
Analysis of current meteorological features versus targets shifted 24 hours ahead (`target_<metric> = shift(-24)`).


In [ ]:
target_corrs = df[["temp_c", "humidity_pct", "pressure_hpa", "wind_speed", "aqi", "target_temp_c", "target_humidity_pct", "target_wind_speed", "target_aqi"]].corr()
plt.figure(figsize=(10, 8))
sns.heatmap(target_corrs, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation Matrix: Present Observations vs 24h Ahead Targets")
plt.show()
